In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/Youtube05-Shakira.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/Youtube03-LMFAO.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/Youtube02-KatyPerry.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/Youtube01-Psy.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/Youtube04-Eminem.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/youtube-spam-collection-v1/Youtube05-Shakira.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/youtube-spam-collection-v1/Youtube03-LMFAO.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/youtube-spam-collection-v1/Youtube02-KatyPerry.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/youtube-spam-collection-v1/Youtube01-Psy.csv
/kaggle/input/datasets/kaustubh129/youtube-spam-collection-data/youtube-spam-collection-v1/Youtube04-Eminem.csv


In [2]:
import pandas as pd
import numpy as np
import re
import os
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.metrics import classification_report, accuracy_score

print("Step 1: Automatically scanning Kaggle directory for your uploaded CSV files...")

# Automatically scan all input directories to find any CSV files you uploaded
csv_files = []
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        if f.endswith('.csv'):
            csv_files.append(os.path.join(root, f))

# Error Check: Make sure files actually exist before processing
if len(csv_files) == 0:
    raise FileNotFoundError("Could not find any uploaded CSV files. Please check the 'Input' section on your right sidebar to verify the dataset is attached to this notebook session.")

print(f"-> Found {len(csv_files)} files to process.")
for path in csv_files:
    print(f"   Loading: {os.path.basename(path)}")

# Read and merge the files safely using Latin-1 encoding to prevent character crashes
df_list = [pd.read_csv(file, encoding='latin1') for file in csv_files]
df = pd.concat(df_list, axis=0, ignore_index=True)

# Keep only the target data features
df = df[['CONTENT', 'CLASS']]
df.columns = ['text', 'label']
print(f"-> Success! Successfully loaded {len(df)} rows from your dataset.")


print("\nStep 2: Cleaning text formatting...")
def clean_comment(text):
    text = str(text).lower()
    # Mask links out so they are standardized
    text = re.sub(r'https?://\s*\S+|www\.\s*\S+', 'url_token', text)
    # Clear redundant spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['cleaned_text'] = df['text'].apply(clean_comment)
print("-> Cleaning complete.")


print("\nStep 3: Training the Cosine Similarity Baseline Model...")
# Train-test split (80% training data, 20% validation test data)
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])

# Feature extraction via TF-IDF (using unigrams and bigrams)
vectorizer = TfidfVectorizer(ngram_range=(1, 2), max_features=1000)
X_train_vectors = vectorizer.fit_transform(train_df['cleaned_text'])
X_test_vectors = vectorizer.transform(test_df['cleaned_text'])

# Isolate known spam profiles to build our master vector matrix
spam_mask = np.array(train_df['label'] == 1)
spam_vectors = X_train_vectors[spam_mask]

# Reshape the mean matrix cleanly to a 2D array for scikit-learn compatibility
average_spam_vector = np.asarray(spam_vectors.mean(axis=0)).reshape(1, -1)

# Evaluate test distances using Cosine Similarity
similarity_scores = cosine_similarity(X_test_vectors, average_spam_vector).flatten()

# Classify comment as spam (1) if it strongly matches our archetype vector
predictions = [1 if score > 0.15 else 0 for score in similarity_scores]
print("-> Evaluation complete.")


print("\n=============================================")
print(f"BASELINE MODEL ACCURACY: {accuracy_score(test_df['label'], predictions) * 100:.2f}%")
print("=============================================\n")
print(classification_report(test_df['label'], predictions, target_names=['Ham', 'Spam']))

Step 1: Automatically scanning Kaggle directory for your uploaded CSV files...
-> Found 10 files to process.
   Loading: Youtube05-Shakira.csv
   Loading: Youtube03-LMFAO.csv
   Loading: Youtube02-KatyPerry.csv
   Loading: Youtube01-Psy.csv
   Loading: Youtube04-Eminem.csv
   Loading: Youtube05-Shakira.csv
   Loading: Youtube03-LMFAO.csv
   Loading: Youtube02-KatyPerry.csv
   Loading: Youtube01-Psy.csv
   Loading: Youtube04-Eminem.csv
-> Success! Successfully loaded 3912 rows from your dataset.

Step 2: Cleaning text formatting...
-> Cleaning complete.

Step 3: Training the Cosine Similarity Baseline Model...
-> Evaluation complete.

BASELINE MODEL ACCURACY: 86.21%

              precision    recall  f1-score   support

         Ham       0.83      0.90      0.86       381
        Spam       0.89      0.83      0.86       402

    accuracy                           0.86       783
   macro avg       0.86      0.86      0.86       783
weighted avg       0.86      0.86      0.86       783

In [3]:
import os
import numpy as np
import torch
from transformers import AlbertTokenizer, AlbertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report

# Suppress the unauthenticated Hugging Face warnings to keep the logs completely clean
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Step 1: Initializing ALBERT Tokenizer and Model architecture...")

# Check for GPU acceleration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"-> Using hardware accelerator: {device.upper()}")

# Load pre-trained ALBERT from Hugging Face
model_name = "albert-base-v2"
tokenizer = AlbertTokenizer.from_pretrained(model_name)
model = AlbertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# 2. Convert your train and test data splits into the Hugging Face format
train_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'label']].rename(columns={'cleaned_text': 'text'}))
test_dataset = Dataset.from_pandas(test_df[['cleaned_text', 'label']].rename(columns={'cleaned_text': 'text'}))

# Tokenization function to map text to input IDs
def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("-> Tokenizing the text data arrays...")
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# 3. Setup training parameters
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          # Reads the data 3 times to tune weights
    weight_decay=0.01,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none"             
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=lambda eval_pred: {
        'accuracy': accuracy_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=1))
    }
)

print("\nStep 2: Training ALBERT on your dataset (This will take about 1-2 minutes)...")
trainer.train()

print("\nStep 3: Calculating final deep learning classifications...")
raw_predictions = trainer.predict(test_tokenized)
albert_predictions = np.argmax(raw_predictions.predictions, axis=1)

print("\n=============================================")
print(f"ALBERT MODEL ACCURACY: {accuracy_score(test_df['label'], albert_predictions) * 100:.2f}%")
print("=============================================\n")
print(classification_report(test_df['label'], albert_predictions, target_names=['Ham', 'Spam']))

Step 1: Initializing ALBERT Tokenizer and Model architecture...
-> Using hardware accelerator: CUDA


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

...

AlbertForSequenceClassification LOAD REPORT from: albert-base-v2
Key                          | Status     | 
-----------------------------+------------+-
predictions.dense.bias       | UNEXPECTED | 
predictions.LayerNorm.bias   | UNEXPECTED | 
predictions.bias             | UNEXPECTED | 
predictions.decoder.bias     | UNEXPECTED | 
predictions.dense.weight     | UNEXPECTED | 
predictions.LayerNorm.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
classifier.weight            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


-> Tokenizing the text data arrays...


Map:   0%|          | 0/3129 [00:00<?, ? examples/s]

Map:   0%|          | 0/783 [00:00<?, ? examples/s]

TypeError: TrainingArguments.__init__() got an unexpected keyword argument 'evaluation_strategy'

In [ ]:
import os
import numpy as np
import torch
from transformers import AlbertTokenizer, AlbertForSequenceClassification, Trainer, TrainingArguments
from datasets import Dataset
from sklearn.metrics import accuracy_score, classification_report

# Suppress the unauthenticated Hugging Face warnings to keep logs clean
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"

print("Step 1: Initializing ALBERT Tokenizer and Model architecture...")

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"-> Using hardware accelerator: {device.upper()}")

model_name = "albert-base-v2"
tokenizer = AlbertTokenizer.from_pretrained(model_name)
model = AlbertForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)

# Convert your train and test data splits into Hugging Face format
train_dataset = Dataset.from_pandas(train_df[['cleaned_text', 'label']].rename(columns={'cleaned_text': 'text'}))
test_dataset = Dataset.from_pandas(test_df[['cleaned_text', 'label']].rename(columns={'cleaned_text': 'text'}))

def tokenize_function(examples):
    return tokenizer(examples['text'], padding="max_length", truncation=True, max_length=128)

print("-> Tokenizing the text data arrays...")
train_tokenized = train_dataset.map(tokenize_function, batched=True)
test_tokenized = test_dataset.map(tokenize_function, batched=True)

# 3. Setup training parameters (FIXED: changed evaluation_strategy to eval_strategy)
training_args = TrainingArguments(
    output_dir="./results",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=3,          
    weight_decay=0.01,
    eval_strategy="epoch",       # Fixed keyword for newer transformers versions
    save_strategy="epoch",
    load_best_model_at_end=True,
    logging_steps=20,
    report_to="none"             
)

# 4. Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=test_tokenized,
    compute_metrics=lambda eval_pred: {
        'accuracy': accuracy_score(eval_pred.label_ids, np.argmax(eval_pred.predictions, axis=1))
    }
)

print("\nStep 2: Training ALBERT on your dataset (This will take about 1-2 minutes)...")
trainer.train()

print("\nStep 3: Calculating final deep learning classifications...")
raw_predictions = trainer.predict(test_tokenized)
albert_predictions = np.argmax(raw_predictions.predictions, axis=1)

print("\n=============================================")
print(f"ALBERT MODEL ACCURACY: {accuracy_score(test_df['label'], albert_predictions) * 100:.2f}%")
print("=============================================\n")
print(classification_report(test_df['label'], albert_predictions, target_names=['Ham', 'Spam']))

In [ ]:
import torch
import re
import numpy as np

# 1. Ensure the model is in evaluation mode
model.eval()

def predict_comment(raw_comment):
    # 2. Clean the input text exactly how we cleaned the training data
    cleaned = str(raw_comment).lower()
    cleaned = re.sub(r'https?://\s*\S+|www\.\s*\S+', 'url_token', cleaned)
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()
    
    # 3. Tokenize the input text
    inputs = tokenizer(cleaned, padding="max_length", truncation=True, max_length=128, return_tensors="pt")
    
    # Move inputs to the same hardware as the model (GPU/CUDA)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # 4. Pass through the model without calculating gradients (saves memory/time)
    with torch.no_grad():
        outputs = model(**inputs)
    
    # 5. Extract probabilities and pick the highest score
    logits = outputs.logits.detach().cpu().numpy()
    prediction_class = np.argmax(logits, axis=1)[0]
    
    # Map the numerical prediction back to a human-readable string
    result = "🚨 SPAM / BOT DETECTED" if prediction_class == 1 else "✅ LEGITIMATE COMMENT (HAM)"
    return result

print("Prediction engine successfully activated! Ready to test input.")

In [ ]:
# Interactive testing loop
print("--- INSTAGRAM SPAM DETECTOR LIVE TEST ---")
print("Type your comment below to test. Type 'exit' to stop.\n")

while True:
    user_input = input("Enter a comment to analyze: ")
    if user_input.lower() == 'exit':
        print("Live test stopped.")
        break
        
    if user_input.strip() == "":
        continue
        
    # Run the prediction
    label = predict_comment(user_input)
    
    print(f"Comment: \"{user_input}\"")
    print(f"Prediction: {label}")
    print("-" * 50)